In [12]:
import os
import numpy as np
import pandas as pd
import json
import systeme as sys
import Allocation
from simulation import simulation
import time

import pickle
import random
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, make_scorer, hamming_loss
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.base import BaseEstimator
from functools import partial

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

instance_name = "G"

if instance_name == "K0":
    test_split_scenarios = [23, 16, 7, 10, 26, 17, 6, 20, 11]
    fms_path = 'fms/3C7R5F.json'
    nombre_de_cellules = 3
    nombre_de_scenarios = 28
else :
    test_split_scenarios = [37, 13, 31, 40, 25, 14, 6, 12, 4, 7, 28, 15, 17]
    fms_path = 'fms/5C14R5F.json'
    nombre_de_cellules = 5
    nombre_de_scenarios = 40

with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
system = sys.systeme(dic)

## generation de data

### génération de data

In [13]:
verbose = 1
ds = instance_name
scenarios_path_prefixe = f"scenarios/{ds}/"
solutions_path_prefixe = f"solution/{ds}_upgraded/"
with open(fms_path, 'r') as json_file:
    dic = json.load(json_file)
system = sys.systeme(dic)
for scenario_path, solution_path in zip(sorted(os.listdir(scenarios_path_prefixe), key= lambda k : int(k[1:].split(".csv")[0])), sorted(os.listdir(solutions_path_prefixe), key= lambda k : int(k.split(".csv")[0].split("_s")[-1]))):
    if verbose > 0:
        print(f"scenario : {scenario_path}",end="\r")
    #initialisation de solution systeme et scenario
    solution = pd.read_csv(solutions_path_prefixe+solution_path, sep=";", index_col=None, header=None).iloc[:,1:-1]
    scenario = pd.DataFrame(np.nan_to_num(pd.read_csv(scenarios_path_prefixe+scenario_path,header=None, index_col=None, sep=";"), nan=0)).astype(int)
    sim = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))], saving=True)
    logs = sim.get_logs()
    if not os.path.exists(f"generated_data_per_family/initial/{ds}/single_label"):
        os.makedirs(f"generated_data_per_family/initial/{ds}/single_label")
    [df.to_csv(f"generated_data_per_family/initial/{ds}/single_label/singlelabel_data_sortedlabels_{scenario_path.split(".")[0]}_cell_{i+1}.csv", sep=";", index=False) for df,i in zip(logs, range(len(logs)))]

### generation de data multilabel

In [14]:
def get_voisins(df:pd.DataFrame, line:int, col:int, system:sys.systeme):
    offset = sum([len(x.ressources) for x in system.cellules[:col]])+1
    arr = np.arange(offset,offset+len(system.cellules[col].ressources))
    voisins_locaux = np.delete(arr, np.where(arr == df.iloc[line, col])[0])
    voisins = []
    for voisin in voisins_locaux:
        df_copy = df.copy()
        df_copy.iloc[line, col] = voisin
        voisins.append(df_copy)
    return voisins

def get_echanges(df:pd.DataFrame, line:int, col:int):
    arr_solution = df.values
    valeur_courante = arr_solution[line, col]

    lignes_suivantes = np.arange(line + 1, arr_solution.shape[0])
    valeurs_suivantes = arr_solution[lignes_suivantes, col]

    mask_diff = valeurs_suivantes != valeur_courante
    lignes_differentes = lignes_suivantes[mask_diff]
    valeurs_differentes = valeurs_suivantes[mask_diff]

    if len(lignes_differentes) == 0:
        return []

    echanges_array = np.repeat(arr_solution[np.newaxis, :, :], len(lignes_differentes), axis=0)
    echanges_array[:, line, col] = valeurs_differentes
    echanges_array[np.arange(len(lignes_differentes)), lignes_differentes, col] = valeur_courante

    return [pd.DataFrame(echange, columns=df.columns) for echange in echanges_array]

def upgrade_insert(system:sys.systeme, scenario:pd.DataFrame, solution:pd.DataFrame, stats:dict, verbose:int):
    symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
    symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
    symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
    produit, cellule = 0,0
    upgraded = False
    old_mct = best_mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))]).mean_completion_time()
    while cellule < solution.shape[1]:
        while produit < solution.shape[0]:
            reset = False
            solutions_generees = get_voisins(solution, line=produit, col=cellule, system=system)
            iteration = 0
            for voisin in solutions_generees:
                if verbose > 0:
                    print(f"\tline {produit+1}, col {cellule+1}/{solution.shape[1]}, iteration {iteration+1}/{len(solutions_generees)}                              ",end="\r")
                try:
                    mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, voisin) for _ in range(len(system.cellules))]).mean_completion_time()
                except Exception as e:
                    if verbose >0:
                        print("changement non autorisé : ",e)
                    iteration+=1
                    continue
                if (mct == best_mct):
                    if verbose > 1:
                        print(f"\n\t\tsymetrie produit {produit+1} cell {cellule+1} solution originale {solution.iloc[produit, cellule]} avec solution {voisin.iloc[produit, cellule]}")
                    symetries_indices[produit, cellule].append(iteration)
                    if stats is not None:
                        stats["symetries"][1,cellule] += 1
                elif (mct < best_mct):
                    if verbose > 0:
                        print(f"new best solution found, from {best_mct} to {mct}, redo all")
                    if stats is not None:
                        stats["improvements"] +=1
                    best_mct = mct
                    solution = voisin
                    reset = True
                    upgraded = True
                    if stats is not None:
                        stats["symetries"] = np.zeros_like(stats["symetries"])
                    break
                iteration+=1
            if reset : 
                cellule, produit = 0, 0
                symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
                symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
                symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
            else :
                produit+=1
        cellule+=1
        produit = 0

    return solution, upgraded, old_mct, best_mct, symetries_indices

def upgrade_swap(system:sys.systeme, scenario:pd.DataFrame, solution:pd.DataFrame, stats:dict, verbose:int):
    symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
    symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
    symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
    produit, cellule = 0,0
    upgraded = False
    old_mct = best_mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution) for _ in range(len(system.cellules))]).mean_completion_time()
    while cellule < solution.shape[1]:
        while produit < solution.shape[0]:
            reset = False
            solutions_generees = get_echanges(solution, produit, cellule)
            iteration = 0
            for i in range(len(solutions_generees)):
                if verbose > 0:
                    print(f"\tline {produit+1}, col {cellule+1}/{solution.shape[1]}, iteration {iteration+1}/{len(solutions_generees)}                              ",end="\r")
                try:
                    mct = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solutions_generees[i]) for _ in range(len(system.cellules))]).mean_completion_time()
                except Exception as e:
                    if verbose > 0 :
                        print("changement non autorisé : ",e)
                    iteration+=1
                    continue
                if (mct == best_mct):
                    if verbose > 1:
                        print(f"\n\t\tsymetrie produit {produit+1} cell {cellule+1} solution originale {solution.iloc[produit, cellule]} avec solution {solutions_generees[i].iloc[produit, cellule]}")
                    symetries_indices[produit, cellule].append(iteration)
                    if stats is not None:
                        stats["symetries"][0,cellule] += 1
                elif (mct < best_mct):
                    if verbose > 0:
                        print(f"\n\t\tnew best solution found, from {best_mct} to {mct}, redo all\n")
                    if stats is not None:
                        stats["improvements"] +=1
                    best_mct = mct
                    solution = solutions_generees[i]
                    reset = True
                    upgraded = True
                    if stats is not None:
                        stats["symetries"] = np.zeros_like(stats["symetries"])
                    break
                iteration+=1

            if reset : 
                cellule, produit = 0, 0
                symetries_indices  = np.empty((solution.shape[0]*solution.shape[1]), dtype=object)
                symetries_indices[:] = [[] for _ in range(solution.shape[0] * solution.shape[1])]
                symetries_indices = symetries_indices.reshape(solution.shape[0], solution.shape[1])
            else :
                produit+=1
        cellule+=1
        produit = 0

    return solution, upgraded, old_mct, best_mct, symetries_indices

def alter_multilabel_dataset_symetries(system:sys.systeme, solution:pd.DataFrame, swapORinsert:int, multilabel_logs:pd.DataFrame, indices_symetrie:np.ndarray, verbose:int):
    for produit in range(indices_symetrie.shape[0]):
        for cellule in range(indices_symetrie.shape[1]):
            if swapORinsert == 0:
                solutions_generees = get_echanges(solution, produit, cellule)
            elif swapORinsert == 1:
                solutions_generees = get_voisins(solution, produit, cellule, system)
            for iteration,i in zip(indices_symetrie[produit,cellule], range(len(indices_symetrie[produit,cellule]))):
                if verbose >0:
                    print(f"\t\tline {produit+1}/{indices_symetrie.shape[0]}, col {cellule+1}/{indices_symetrie.shape[1]}, iteration {i+1}/{len(indices_symetrie[produit,cellule])}",end="\r")
                multilabel_logs[cellule].iloc[produit, solutions_generees[iteration].iloc[produit, cellule]-1 - sum([len(c.ressources) for c in system.cellules][:cellule])-len(system.cellules[cellule].ressources)] = 1  
    return multilabel_logs


In [15]:
verbose = 1
ds = instance_name
scenarios_path_prefixe = f"scenarios/{ds}/"
solutions_path_prefixe = f"solution/{ds}_best/"
upgraded_solutions_path_prefixe = f"solution/{ds}_upgraded/"

with open(fms_path, 'r') as json_file: #5C14R5F #3C7R5F
    dic = json.load(json_file)
system = sys.systeme(dic)

for scenario_path, solution_path in zip(sorted(os.listdir(scenarios_path_prefixe), key= lambda k : int(k[1:].split(".csv")[0])), sorted(os.listdir(solutions_path_prefixe), key= lambda k : int(k.split(".csv")[0].split("_s")[-1]))):
    if verbose > -1:
        print(f"\nscenario : {scenario_path}")
    #initialisation de solution systeme et scenario
    solution = pd.read_csv(upgraded_solutions_path_prefixe+solution_path, sep=";", index_col=None, header=None).iloc[:,1:-1]
    scenario = pd.DataFrame(np.nan_to_num(pd.read_csv(scenarios_path_prefixe+scenario_path, index_col=None, sep=";", header=None), nan=0)).astype(int)
    reset = True
    start_time = time.time()
    while reset:
        solution, upgraded, old_mct, new_mct, symetries_indices_swap = upgrade_swap(system, scenario, solution, None, verbose)
        if verbose > 0:
            print(f"\n\n\tswap improved ? {upgraded} " + (f"from {old_mct} to {new_mct}" if upgraded else f" with {old_mct}"))
        solution, upgraded, old_mct, new_mct, symetries_indices_insert = upgrade_insert(system, scenario, solution, None, verbose)
        if verbose > 0:
            print(f"\n\n\tinsert improved ? {upgraded} " + (f"from {old_mct} to {new_mct}" if upgraded else f" with {old_mct}"))
        reset = upgraded
    # arrivés là on est surs d'avoir la meilleure solution non ameliorable dans solution, on sauvegarde la solution
    solution.insert(0, None, [f"P{i+1}" for i in range(len(solution))])
    solution[len(solution.columns)] = 0
    solution.to_csv(upgraded_solutions_path_prefixe+solution_path, sep=";", index=False, header=False)

    # symetries
    if verbose > 0:
        print(f"\n\t meilleure solution trouvée, début traitement symétries\n")
    base_logs = simulation(system=system, scenario=scenario, allocators=[Allocation.StaticAllocator(system, solution.iloc[:,1:-1]) for _ in range(len(system.cellules))], saving=True).get_logs()
    expected_classes = [[0,1],[0,1,2],[0,1]] if instance_name == "K0" else [[0,1],[0,1,2],[0,1,2,3],[0,1,2],[0,1]]

    # Transformation de base_logs
    multilabel_logs = []
    for i, base_log in enumerate(base_logs):
        selected_column = base_log.columns[-1]
        dummies = pd.get_dummies(base_log, columns=[selected_column], drop_first=False, dtype=int)

        # S'assurer que toutes les colonnes attendues sont présentes
        for cls in expected_classes[i]:
            col_name = selected_column +"_"+ str(cls)+".0"
            if col_name not in dummies.columns:
                dummies[col_name] = 0  # Ajouter la colonne si elle est absente

        # Réorganiser l'ordre des colonnes
        cols = list(base_log.columns[:-1]) + [selected_column +"_"+ str(cls)+".0" for cls in expected_classes[i]]
        dummies = dummies[cols]  
        
        new_base_log = dummies
        
        multilabel_logs.append(new_base_log)

    if not os.path.exists(f"generated_data_per_family/initial/{ds}/multilabel/none"):
        os.makedirs(f"generated_data_per_family/initial/{ds}/multilabel/none")
    if not os.path.exists(f"generated_data_per_family/initial/{ds}/multilabel/swap"):
        os.makedirs(f"generated_data_per_family/initial/{ds}/multilabel/swap")
    if not os.path.exists(f"generated_data_per_family/initial/{ds}/multilabel/insert"):
        os.makedirs(f"generated_data_per_family/initial/{ds}/multilabel/insert")
    if not os.path.exists(f"generated_data_per_family/initial/{ds}/multilabel/machine_id"):
        os.makedirs(f"generated_data_per_family/initial/{ds}/multilabel/machine_id")
    
    for cell_log, i in zip(multilabel_logs, range(len(multilabel_logs))): # solution opti locale sans symetries
        cell_log.to_csv(f"generated_data_per_family/initial/{ds}/multilabel/none/multilabel_data_original_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)
    
    multilabel_logs_swap = alter_multilabel_dataset_symetries(system, solution.iloc[:,1:-1], 0, multilabel_logs, symetries_indices_swap, verbose)
    for cell_log, i in zip(multilabel_logs_swap, range(len(multilabel_logs_swap))): # avec swap seul
        cell_log.to_csv(f"generated_data_per_family/initial/{ds}/multilabel/swap/multilabel_data_swap_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)

    multilabel_logs_insert = alter_multilabel_dataset_symetries(system, solution.iloc[:,1:-1], 1, multilabel_logs, symetries_indices_insert, verbose)  
    for cell_log, i in zip(multilabel_logs_insert, range(len(multilabel_logs_insert))): # avec insert seul
        cell_log.to_csv(f"generated_data_per_family/initial/{ds}/multilabel/insert/multilabel_data_insert_{scenario_path.split(".")[0]}_cell_{i + 1}.csv", sep=";", index=None)


scenario : s1.csv
	line 99, col 5/5, iteration 1/1                                

	swap improved ? False  with 296.5
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 296.5

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 2/20
scenario : s2.csv
	line 99, col 5/5, iteration 1/1                                

	swap improved ? False  with 305.71
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 305.71

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 1/10
scenario : s3.csv
	line 96, col 5/5, iteration 4/4                                

	swap improved ? False  with 303.49
	line 100, col 5/5, iteration 1/1                              

	insert improved ? False  with 303.49

	 meilleure solution trouvée, début traitement symétries

		line 100/100, col 4/5, iteration 1/12
scenario : s4.csv
	line 99, col 5/5

In [16]:
# R1 et R2 dans la meme cellule  ---
# R1 tps setup R2 tps setup   ---
# R1 tps process R2 tps process  ---
# R1 pred_fam == R2 pred_fam 
# R1 current charge == R2 current charge


# 1 localliser les ressources identiques dans systeme pour chaque cellule
ds = instance_name
data_path_prefixe = f"generated_data_per_family/initial/{ds}/multilabel/none/"
data_path_save = f"generated_data_per_family/initial/{ds}/multilabel/machine_id/"
with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
system = sys.systeme(dic)
nb_symetries = {}
nb_cells = len(system.cellules)
identiques = []
for c in system.cellules:
    ids_cell = []
    for r in c.ressources:
        ids_ress = []
        for r2 in c.ressources:
            if r is r2 or (r.processTimes == r2.processTimes and r.setupTimes == r2.setupTimes) :
                ids_ress.append(True)
            else :
                ids_ress.append(False)
        ids_cell.append(ids_ress)
    identiques.append(ids_cell)
identical_couples = [[(i+1, j+1) for i, j in np.argwhere((np.array(idd).astype(int) == 1) & np.triu(np.ones(np.array(idd).astype(int).shape, dtype=bool), k=1))] for idd in identiques]
cell_containing_identities = np.where(np.array([int(np.sum(x)/len(x)) for x in identiques]) != 1)[0]

# 2 pour chaque dataset d'un scenario et d'une cellule contenant des identités
for dataset_path in os.listdir(data_path_prefixe):
    if dataset_path.endswith("json"):
        continue
    print(f"\nscenario {dataset_path.split("_cell")[0].split("_s")[-1]} :")
    cell = int(dataset_path[:-4].split("_")[-1]) -1

    nb_symetries[f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv"] = nb_symetries.get(f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv", [0]*nb_cells)
    if cell in cell_containing_identities:
        print(f"\tcellule {cell+1} :")
        dataset = pd.read_csv(data_path_prefixe+dataset_path, sep=";", index_col=None)
        
        # 3 trouver les colonnes a verifier
        for line in range(dataset.shape[0]):
            for couple in identical_couples[cell]:
                # 4 verifier les conditions
                if dataset["predFamilies_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[0])][line] == dataset["predFamilies_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[1])][line]  and  dataset["Cur_Charge_comparable_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[0])][line] == dataset["Cur_Charge_comparable_R"+str(sum([len(sl.ressources) for sl in system.cellules[:cell+1][:-1]])+couple[1])][line] \
                and  ((dataset[f"Selected Resource_{couple[0]-1}.0"][line] + dataset[f"Selected Resource_{couple[1]-1}.0"][line]) == 1):
                    print("\t\tsymetrie ligne : ", line)
                    nb_symetries[f"s{dataset_path.split("_cell")[0].split("_s")[-1]}.csv"][cell] += 1
                    # 5 succes, symetrie detectee rajouter les annotations
                    dataset.loc[line, f"Selected Resource_{couple[0]-1}.0"] = 1
                    dataset.loc[line, f"Selected Resource_{couple[1]-1}.0"] = 1

        # 6 sauvegarder le dataframe
        dataset.to_csv(data_path_save+dataset_path, sep=";", index=None)

#   2   3   2



scenario 10 :
	cellule 1 :
		symetrie ligne :  0

scenario 10 :

scenario 10 :

scenario 10 :
	cellule 4 :
		symetrie ligne :  0
		symetrie ligne :  0
		symetrie ligne :  1
		symetrie ligne :  1
		symetrie ligne :  2
		symetrie ligne :  4
		symetrie ligne :  4
		symetrie ligne :  5
		symetrie ligne :  7
		symetrie ligne :  8
		symetrie ligne :  9
		symetrie ligne :  10
		symetrie ligne :  10
		symetrie ligne :  11
		symetrie ligne :  12
		symetrie ligne :  13
		symetrie ligne :  13
		symetrie ligne :  14
		symetrie ligne :  17
		symetrie ligne :  17
		symetrie ligne :  18
		symetrie ligne :  20
		symetrie ligne :  21
		symetrie ligne :  24
		symetrie ligne :  24
		symetrie ligne :  25
		symetrie ligne :  28
		symetrie ligne :  30
		symetrie ligne :  30
		symetrie ligne :  31
		symetrie ligne :  32
		symetrie ligne :  32
		symetrie ligne :  33
		symetrie ligne :  33
		symetrie ligne :  34
		symetrie ligne :  34
		symetrie ligne :  35
		symetrie ligne :  37
		symetrie ligne :  37
		syme

#### concatenage des multilabel en un dataset final multilabel

In [17]:

if not os.path.exists(f"generated_data_per_family/initial/{ds}/multilabel/final"):
    os.makedirs(f"generated_data_per_family/initial/{ds}/multilabel/final")

destination_path = f"generated_data_per_family/initial/{ds}/multilabel/final/"
source_none_path = f"generated_data_per_family/initial/{ds}/multilabel/none/"
source_swap_path = f"generated_data_per_family/initial/{ds}/multilabel/swap/"
source_insert_path = f"generated_data_per_family/initial/{ds}/multilabel/insert/"
source_mid_path = f"generated_data_per_family/initial/{ds}/multilabel/machine_id/"

swap_files = [source_swap_path+f for f in os.listdir(source_swap_path)]
insert_files = [source_insert_path+f for f in os.listdir(source_insert_path)]
mid_files = [source_mid_path+f for f in os.listdir(source_mid_path)]



nb_cells = nombre_de_cellules
nb_scenarios = 28 if instance_name == "K0" else 40

for c in range(1,1+nb_cells):
    for scen in range(1,nb_scenarios+1):
        df_swap = pd.read_csv(source_swap_path+f"multilabel_data_swap_s{scen}_cell_{c}.csv", sep=";")
        df_insert = pd.read_csv(source_insert_path+f"multilabel_data_insert_s{scen}_cell_{c}.csv", sep=";")
        if source_mid_path+f"multilabel_data_original_s{scen}_cell_{c}.csv" in mid_files:
            df_mid = pd.read_csv(source_mid_path+f"multilabel_data_original_s{scen}_cell_{c}.csv", sep=";")
        else:
            df_mid = pd.read_csv(source_none_path+f"multilabel_data_original_s{scen}_cell_{c}.csv", sep=";")
        
        df_final = df_swap.copy()
        selected_cols = [col for col in df_final.columns if col.startswith("Selected")]
        df_final[selected_cols] = (df_swap[selected_cols] + df_insert[selected_cols] + df_mid[selected_cols]).clip(upper=1)
        
        df_final.to_csv(destination_path+f"multilabel_data_s{scen}_cell_{c}.csv", sep=";", index=None)

### séparation par familles

In [18]:
nb_familles = 5

for c in range(1,1+nb_cells):
    for scen in range(1,nb_scenarios+1):
        df_all = pd.read_csv(f"generated_data_per_family/initial/{ds}/multilabel/final/multilabel_data_s{scen}_cell_{c}.csv", sep=";")
        df_all_s = pd.read_csv(f"generated_data_per_family/initial/{ds}/single_label/singlelabel_data_sortedlabels_s{scen}_cell_{c}.csv", sep=";")
        df_par_famille = {}
        df_par_famille_s = {}

        for famille in df_all["Family"].unique():
            df_famille = df_all[df_all['Family'] == famille].copy()
            df_famille.drop(columns=['Family'], inplace=True)
            df_par_famille[int(famille)] = df_famille
        
        for famille in df_all_s["Family"].unique():
            df_famille_s = df_all_s[df_all_s['Family'] == famille].copy()
            df_famille_s.drop(columns=['Family'], inplace=True)
            df_par_famille_s[int(famille)] = df_famille_s

        #saving 
        for ff in range(1,6):
            if not os.path.exists(f"generated_data_per_family/final/{ds}/multilabel/f{ff}"):
                os.makedirs(f"generated_data_per_family/final/{ds}/multilabel/f{ff}")
        
            if not os.path.exists(f"generated_data_per_family/final/{ds}/single_label/f{ff}"):
                os.makedirs(f"generated_data_per_family/final/{ds}/single_label/f{ff}")
        
            df_par_famille[ff].to_csv(f"generated_data_per_family/final/{ds}/multilabel/f{ff}/multilabel_data_s{scen}_cell_{c}.csv", sep=";", index=None)
            df_par_famille_s[ff].to_csv(f"generated_data_per_family/final/{ds}/single_label/f{ff}/singlelabel_data_s{scen}_cell_{c}.csv", sep=";", index=None)


## entrainement

### singlelabel

In [19]:
def calcul_gap(modele, scaler, cellule, scenarios, famille):
    scenarios_path = f"scenarios/{instance_name}"
    solution_path = f"solution/{instance_name}_upgraded//"
    
    with open(fms_path, 'r') as json_file:
        dic = json.load(json_file)
    s = sys.systeme(dic)
    all_gaps = []
    for f in os.listdir(scenarios_path):
        if int(f.split(".")[0][1:]) not in scenarios:
            continue
        df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f,header=None, index_col=None, sep=";"), nan=0)).astype(int)
        own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
        if len(own_sol_path_list) == 0:
            continue
        sol_path = own_sol_path_list[0]
        sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

        allocators = [Allocation.StaticAllocator(s, sol) for _ in range(len(s.cellules))]
        ref_mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        
        allocators[cellule] = Allocation.DynamicAllocator(s, Allocation.OneFamilyOnlyModel(modele, famille, s, cellule, sol, True), scaler, to_categorical=True)
        mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        gap = (100*(mct - ref_mct)/ref_mct)
        all_gaps.append(gap)

    return np.mean(all_gaps)

#multilabel
def callback(_, trial, cell, X_train, Y_train, scaler, scenarios_test, scores, cv, famille):
    current_params = trial.params
    current_model = RandomForestClassifier(**current_params, random_state=RANDOM_SEED)
    current_model.fit(X_train, Y_train)

    current_gap = calcul_gap(current_model, scaler, cell, scenarios_test, famille)
    acc = cross_val_score(current_model, X_train, Y_train, cv=cv, scoring="accuracy").mean()
    #scores.append(acc)
    scores.append([current_gap, acc])

#mutlilabel
def objective_rf(trial, X_train, y_train, cv=3, scenarios_test=None, scaler=None, cell=None, famille=None):
    """
    Fonction objectif pour optimiser les hyperparamètres d'un Random Forest avec Optuna.
    """
    # Définir les hyperparamètres à optimiser
    n_estimators = trial.suggest_int("n_estimators", 3, 400)
    max_depth = trial.suggest_int("max_depth", 2, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 25)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 25)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    # Initialiser le modèle avec les hyperparamètres
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=RANDOM_SEED
    )

    # Effectuer une validation croisée pour évaluer la performance

    model.fit(X_train, y_train)

    return  -calcul_gap(model, scaler, cell, scenarios_test, famille)
    #scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    #return scores.mean()  # Retourner la moyenne des scores comme métrique


def train_random_forest_with_optuna(X_train, y_train, X_test, y_test, scaler, max_trials=49, alpha=50, beta=0.1, patience_limit=1, random_seed=RANDOM_SEED, cell=1, famille=0, scenarios_test=[]):
    """
    Entraîner un Random Forest optimisé via une recherche bayésienne sur les hyperparamètres.
    """
    # Initialiser une étude Optuna
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
    total_trials = 0
    remaining_trials = alpha  # Nombre initial de trials
    step = 0
    best_known_score = 0
    patience = 0
    scores = []

    custom_callback = partial(callback, X_train=X_train, Y_train=y_train, scenarios_test=scenarios_test, scores=scores, cell=cell, scaler=scaler, cv=3, famille=famille)

    while total_trials < max_trials and patience < patience_limit:
        print(f"Optimizing: Step {step + 1}, Remaining trials: {remaining_trials}")
        study.optimize(lambda trial: objective_rf(trial, X_train, y_train, cv=3, scenarios_test=scenarios_test, scaler=scaler, cell=cell, famille=famille), n_trials=remaining_trials, callbacks=[custom_callback])
        
        remaining_trials = int(np.ceil(alpha / (1 + beta * step)))
        total_trials += remaining_trials
        step += 1

        best_current_score = study.best_value

        if best_current_score > best_known_score:
            best_known_score = best_current_score
            patience = 0 
        else:
            patience += 1  

    # Afficher les meilleurs paramètres trouvés
    best_params = study.best_params
    print(f"Best Hyperparameters: {best_params}")

    # Entraîner le modèle avec les meilleurs hyperparamètres
    model = RandomForestClassifier(**best_params, random_state=random_seed)
    model.fit(X_train, y_train)

    # Prédire sur le jeu de test et afficher les performances
    predictions = model.predict(X_test)
    print("Optimized Random Forest - Classification Report:")
    print(classification_report(y_test, predictions))


    return model, best_params


In [20]:
with open(fms_path, 'r') as json_file: #data preparation for training
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path_main = f"generated_data_per_family/final/{instance_name}/single_label/"
test_scenario_count = nombre_de_scenarios//3
dfs_train_f, dfs_test_f = {}, {}

for fam in range(1,6):
    data_folder_path = data_folder_path_main + f"f{fam}/"
    file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]
    for cell,c in zip(s.cellules,range(len(s.cellules))):
        file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:] if file.endswith(f"cell_{c+1}.csv")])
        kept_cols = cell.header[:-1]
        columns_per_cell.append(kept_cols)
        
        if c == 0:
            test_split = list(range(1, len(file_names_per_cell[0])+1))
            random.shuffle(test_split)
            test_split = test_split[:test_scenario_count]
            test_split_scenarios = test_split[:len(test_split)]
            train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

        paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

        filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
        filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
        
        df_train = pd.concat(filtered_dfs_train, ignore_index=True)
        df_train.fillna(0, inplace=True)
        dfs_train.append(df_train.astype(float))

        df_test = pd.concat(filtered_dfs_test, ignore_index=True)
        df_test.fillna(0, inplace=True)
        dfs_test.append(df_test.astype(float))
    dfs_test_f[fam] = dfs_test
    dfs_train_f[fam] = dfs_train


### models training

In [21]:
#training (les deux datasets le meme code, changer juste point rouge)

models_f = {}

for fam in range(1,6):
    models = []
    for i, (train_df, test_df) in enumerate(zip(dfs_train_f[fam], dfs_test_f[fam])):

        print(f"\nProcessing famille {fam} dataset cellule {i+1}...")
        nb_classes = len([col for col in train_df.columns if col.startswith("Selected")])
        
        X_train, y_train = train_df.iloc[:, :-nb_classes], train_df.iloc[:, -nb_classes:].astype(int)
        X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].astype(int)

        model = train_random_forest_with_optuna(X_train, y_train, X_test, y_test, None, max_trials = 1000, alpha= 100, beta = 0.1, random_seed=RANDOM_SEED, scenarios_test=test_split_scenarios,cell=i, famille=fam)
        
        predictions_test = model[0].predict(X_test)
        print(f"Validation Performance for famille {fam} dataset cellule {i+1}:\n",classification_report(y_test, predictions_test))
        print(f"- - - saving - - -")
        models.append(model[0])

        if not os.path.exists(f"generated_models_family/{instance_name}/singlelabel/models_cell{i+1}"):
            os.makedirs(f"generated_models_family/{instance_name}/singlelabel/models_cell{i+1}")
        
        with open(f"generated_models_family/{instance_name}/singlelabel/models_cell{i+1}/standard_RandomForest_f{fam}.pkl", 'wb') as file: #change G for K0
            pickle.dump(model[0], file)
    models_f[fam] = models
    

[I 2025-07-18 05:46:05,470] A new study created in memory with name: no-name-46b6f194-4ea2-4e78-84ae-d6aef7c9f11e



Processing famille 1 dataset cellule 1...
Optimizing: Step 1, Remaining trials: 100


[I 2025-07-18 05:46:08,615] Trial 0 finished with value: -16.27833766839554 and parameters: {'n_estimators': 152, 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: -16.27833766839554.
[I 2025-07-18 05:46:16,008] Trial 1 finished with value: -8.644684107736394 and parameters: {'n_estimators': 347, 'max_depth': 19, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -8.644684107736394.
[I 2025-07-18 05:46:23,963] Trial 2 finished with value: -17.169155438445394 and parameters: {'n_estimators': 75, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 14, 'max_features': None}. Best is trial 1 with value: -8.644684107736394.
[I 2025-07-18 05:46:30,155] Trial 3 finished with value: -14.88954087686087 and parameters: {'n_estimators': 58, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 1 with value: -8.64468410773

Best Hyperparameters: {'n_estimators': 143, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt'}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.76      0.79       165
           1       0.70      0.76      0.73       120

    accuracy                           0.76       285
   macro avg       0.76      0.76      0.76       285
weighted avg       0.77      0.76      0.76       285

Validation Performance for famille 1 dataset cellule 1:
               precision    recall  f1-score   support

           0       0.81      0.76      0.79       165
           1       0.70      0.76      0.73       120

    accuracy                           0.76       285
   macro avg       0.76      0.76      0.76       285
weighted avg       0.77      0.76      0.76       285

- - - saving - - -

Processing famille 1 dataset cellule 2...
Optimizing: Step 1, Remaining trials: 100


[I 2025-07-18 06:00:29,379] Trial 0 finished with value: -3.3332654395893893 and parameters: {'n_estimators': 152, 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: -3.3332654395893893.
[I 2025-07-18 06:00:38,671] Trial 1 finished with value: -3.0248541825565267 and parameters: {'n_estimators': 347, 'max_depth': 19, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -3.0248541825565267.
[I 2025-07-18 06:00:48,195] Trial 2 finished with value: -3.561454200007804 and parameters: {'n_estimators': 75, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 14, 'max_features': None}. Best is trial 1 with value: -3.0248541825565267.
[I 2025-07-18 06:00:55,624] Trial 3 finished with value: -3.3042830506917706 and parameters: {'n_estimators': 58, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 1 with value: -3.024854

...

Best Hyperparameters: {'n_estimators': 16, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': None}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.86      0.75        66
           1       0.82      0.77      0.79        87
           2       0.67      0.57      0.62        42
           3       0.82      0.58      0.68        31

    accuracy                           0.73       226
   macro avg       0.74      0.70      0.71       226
weighted avg       0.74      0.73      0.73       226

Validation Performance for famille 5 dataset cellule 3:
               precision    recall  f1-score   support

           0       0.66      0.86      0.75        66
           1       0.82      0.77      0.79        87
           2       0.67      0.57      0.62        42
           3       0.82      0.58      0.68        31

    accuracy                           0.73       226
   m

[I 2025-07-18 10:39:09,034] Trial 0 finished with value: -0.03710178710433768 and parameters: {'n_estimators': 152, 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.03710178710433768.
[I 2025-07-18 10:39:15,612] Trial 1 finished with value: -0.03843358584009796 and parameters: {'n_estimators': 347, 'max_depth': 19, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.03710178710433768.
[I 2025-07-18 10:39:22,431] Trial 2 finished with value: -0.03707800458191286 and parameters: {'n_estimators': 75, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 14, 'max_features': None}. Best is trial 2 with value: -0.03707800458191286.
[I 2025-07-18 10:39:27,853] Trial 3 finished with value: -0.03622499859540382 and parameters: {'n_estimators': 58, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 3 with value: -

Best Hyperparameters: {'n_estimators': 5, 'max_depth': 24, 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_features': 'log2'}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.54      0.50        76
           1       0.50      0.44      0.47        80
           2       0.43      0.41      0.42        70

    accuracy                           0.46       226
   macro avg       0.46      0.46      0.46       226
weighted avg       0.47      0.46      0.46       226

Validation Performance for famille 5 dataset cellule 4:
               precision    recall  f1-score   support

           0       0.47      0.54      0.50        76
           1       0.50      0.44      0.47        80
           2       0.43      0.41      0.42        70

    accuracy                           0.46       226
   macro avg       0.46      0.46      0.46       226
weighted avg       0.47      0.46      0.46       226

-

[I 2025-07-18 10:49:21,107] Trial 0 finished with value: -0.26425770242928703 and parameters: {'n_estimators': 152, 'max_depth': 29, 'min_samples_split': 19, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.26425770242928703.
[I 2025-07-18 10:49:27,559] Trial 1 finished with value: -0.25983268078002225 and parameters: {'n_estimators': 347, 'max_depth': 19, 'min_samples_split': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -0.25983268078002225.
[I 2025-07-18 10:49:34,197] Trial 2 finished with value: -0.31893256317479296 and parameters: {'n_estimators': 75, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 14, 'max_features': None}. Best is trial 1 with value: -0.25983268078002225.
[I 2025-07-18 10:49:39,426] Trial 3 finished with value: -0.29207191159866364 and parameters: {'n_estimators': 58, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 1 with value: -

Best Hyperparameters: {'n_estimators': 361, 'max_depth': 25, 'min_samples_split': 25, 'min_samples_leaf': 17, 'max_features': 'log2'}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.89      0.83       110
           1       0.88      0.77      0.82       116

    accuracy                           0.83       226
   macro avg       0.83      0.83      0.83       226
weighted avg       0.83      0.83      0.83       226

Validation Performance for famille 5 dataset cellule 5:
               precision    recall  f1-score   support

           0       0.78      0.89      0.83       110
           1       0.88      0.77      0.82       116

    accuracy                           0.83       226
   macro avg       0.83      0.83      0.83       226
weighted avg       0.83      0.83      0.83       226

- - - saving - - -


## Simulation

In [22]:
models_classes = []
models_not_classes = []
for i in range (1,nombre_de_cellules+1):
    models = []
    for fam in range(1,6):
        with open(f"generated_models_family/{instance_name}/singlelabel/models_cell{i}/standard_RandomForest_f{fam}.pkl", 'rb') as f:
            m = pickle.load(f)
            models.append(m)
    models_not_classes. append(models)
    models_classes.append(Allocation.PerFamilySingleLabel(models, system, i))

scenarios_path = f"scenarios/{instance_name}"
solution_path = f"solution/{instance_name}_upgraded/"

with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
s = sys.systeme(dic)
results = np.zeros((0,4))
reference_results = np.zeros((0,4))

for f in sorted(os.listdir(scenarios_path), key=lambda y: int(y.split(".")[0][1:])):
    #print(f"{f.split(".")[0]}\t:\t", end = "")
    #if int(f.split(".")[0][1:]) not in test_split_scenarios:
        #print("dans les scenarios d'entrainement",end="\r")
    #    continue
    df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f, index_col=None, header=None, sep=";"), nan=0)).astype(int)
    own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
    
    sol_path = own_sol_path_list[0]
    sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

    dyn_allocs_not_classes = [Allocation.DynamicAllocator(s, model_class, None, keep_cols=None, to_categorical=True, singlelabel=True) for model_class in models_not_classes]
    dyn_allocs = [Allocation.DynamicAllocator(s, model_class, None, keep_cols=None, to_categorical=True, singlelabel=True) for model_class in models_classes]
    static_allocators = [Allocation.StaticAllocator(s, sol) for _ in range(nombre_de_cellules)]
    
    allocators = [#static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2]
                  ] if instance_name == "K0" else [
                      #static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2], 
                  #static_allocators[3], 
                  dyn_allocs[3], 
                  #static_allocators[4], 
                  dyn_allocs[4]
                  ]
    
    old_allocators = [#static_allocators[0],
                  dyn_allocs_not_classes[0], 
                  #static_allocators[1], 
                  dyn_allocs_not_classes[1], 
                  #static_allocators[2], 
                  dyn_allocs_not_classes[2]
                  ] if instance_name == "K0" else [
                      #static_allocators[0],
                  dyn_allocs_not_classes[0], 
                  #static_allocators[1], 
                  dyn_allocs_not_classes[1], 
                  #static_allocators[2], 
                  dyn_allocs_not_classes[2], 
                  #static_allocators[3], 
                  dyn_allocs_not_classes[3], 
                  #static_allocators[4], 
                  dyn_allocs_not_classes[4]
                  ]
    
    sim = simulation(system=s, scenario=df, allocators=allocators, labels_encoded=False)
    results = np.vstack((results, np.array([f.split(".")[0], sim.average_flowtime(), sim.mean_completion_time(), sim.total_decision_times()])))

    sim_ref = simulation(system=s, scenario=df, allocators=static_allocators)
    reference_results = np.vstack((reference_results, np.array([f.split(".")[0], sim_ref.average_flowtime(), sim_ref.mean_completion_time(), sim_ref.total_decision_times()])))

    print(f"{f.split(".")[0]}\t{sim.mean_completion_time()}\t{sim_ref.mean_completion_time()}\t{int(f.split(".")[0][1:]) in test_split_scenarios}")
    #print(f"ended with mct : {sim.mean_completion_time()} while the reference mct is {sim_ref.mean_completion_time()}")
    #sim_ref.gantt(path=f"gants/gant_{f.split(".")[0]}_ref.png")
    #sim.gantt(path=f"gants/gant_{f.split(".")[0]}_{total_nb_scenarios//3}.png")

df = pd.DataFrame(results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
df_sorted = df.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

df_ref = pd.DataFrame(reference_results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df_ref['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
ref = df_ref.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

print(f"gap {"ag" if isinstance(allocators[0], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[1], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[2], Allocation.StaticAllocator) else "m"} {(100*(df_sorted - ref)/ref).mean()} %")

s1	339.56	296.5	False
s2	349.26	305.71	False
s3	347.62	303.49	False
s4	343.1	294.63	True
s5	322.49	296.63	False
s6	360.59	297.42	True
s7	350.8	298.24	True
s8	341.67	301.53	False
s9	338.71	312.66	False
s10	332.91	299.32	False
s11	372.26	303.6	False
s12	358.34	309.91	True
s13	329.14	289.79	True
s14	315.42	286.18	True
s15	340.18	290.94	True
s16	357.5	294.05	False
s17	368.62	312.46	True
s18	363.79	302.11	False
s19	345.78	311.99	False
s20	339.51	293.85	False
s21	302.16	284.78	False
s22	368.74	306.75	False
s23	327.57	299.14	False
s24	331.13	292.74	False
s25	324.04	294.01	True
s26	309.53	280.28	False
s27	335.17	301.28	False
s28	332.38	290.57	True
s29	317.96	290.09	False
s30	312.37	287.15	False
s31	310.0	286.95	True
s32	330.38	292.29	False
s33	314.96	284.18	False
s34	336.24	299.02	False
s35	332.78	294.45	False
s36	361.96	287.62	False
s37	326.13	277.46	True
s38	330.02	294.8	False
s39	310.57	310.03	False
s40	310.61	288.4	True
gap m m m 13.492115791112491 %


## Evaluation

In [5]:
with open(fms_path, 'r') as json_file: #data preparation for training
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path_main = f"generated_data_per_family/final/{instance_name}/single_label/"
total_nb_scenarios = 28 * 3
test_scenario_count = total_nb_scenarios//9
dfs_train_f, dfs_test_f = {}, {}

for fam in range(1,6):
    data_folder_path = data_folder_path_main + f"f{fam}/"
    file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]
    for cell,c in zip(s.cellules,range(len(s.cellules))):
        file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:total_nb_scenarios] if file.endswith(f"cell_{c+1}.csv")])
        kept_cols = cell.header[:-1]
        columns_per_cell.append(kept_cols)
        
        if c == 0:
            test_split = list(range(1, len(file_names_per_cell[0])+1))
            random.shuffle(test_split)
            test_split = test_split[:test_scenario_count]
            test_split_scenarios = test_split[:len(test_split)]
            train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

        paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

        filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
        filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
        
        df_train = pd.concat(filtered_dfs_train, ignore_index=True)
        df_train.fillna(0, inplace=True)
        dfs_train.append(df_train.astype(float))

        df_test = pd.concat(filtered_dfs_test, ignore_index=True)
        df_test.fillna(0, inplace=True)
        dfs_test.append(df_test.astype(float))
    dfs_test_f[fam] = dfs_test
    dfs_train_f[fam] = dfs_train

In [9]:
with open(fms_path, 'r') as json_file: #data preparation for training
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path_main = f"generated_data_per_family/final/{instance_name}/multilabel/"
total_nb_scenarios = 28 * 3
test_scenario_count = total_nb_scenarios//9
dfs_train_m, dfs_test_m = {}, {}

for fam in range(1,6):
    data_folder_path = data_folder_path_main + f"f{fam}/"
    file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]
    for cell,c in zip(s.cellules,range(len(s.cellules))):
        file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:total_nb_scenarios] if file.endswith(f"cell_{c+1}.csv")])
        kept_cols = cell.header[:-1]
        columns_per_cell.append(kept_cols)
        
        if c == 0:
            test_split = list(range(1, len(file_names_per_cell[0])+1))
            random.shuffle(test_split)
            test_split = test_split[:test_scenario_count]
            test_split_scenarios = test_split[:len(test_split)]
            train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

        paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

        filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
        filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
        
        df_train = pd.concat(filtered_dfs_train, ignore_index=True)
        df_train.fillna(0, inplace=True)
        dfs_train.append(df_train.astype(float))

        df_test = pd.concat(filtered_dfs_test, ignore_index=True)
        df_test.fillna(0, inplace=True)
        dfs_test.append(df_test.astype(float))
    dfs_test_m[fam] = dfs_test
    dfs_train_m[fam] = dfs_train


models_cell= {}
for i in range (1,4):
    models = []
    for fam in range(1,6):
        with open(f"generated_models_family/{instance_name}/singlelabel/models_cell{i}/standard_RandomForest_f{fam}.pkl", 'rb') as f:
            models.append(pickle.load(f))
    models_cell[i] = models

In [11]:
for i, (models_cellule, tests_df, tests_df_m) in enumerate(zip(models_cell.values(), dfs_test_f.values(), dfs_test_m.values())):

    for fam in range(1,6):
        print(f"\nProcessing dataset cellule {i+1}, famille {fam}...")
        
        model = models_cell[i+1][fam-1]
        test_df = dfs_test_f[fam][i]
        test_df_m = dfs_test_m[fam][i]
        
        nb_classes = len([col for col in test_df.columns if col.startswith("Selected")])

        X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].astype(int)

        nb_classes = len([col for col in test_df_m.columns if col.startswith("Selected")])
        X_test_m, y_test_m = test_df_m.iloc[:, :-nb_classes], test_df_m.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)

        predictions_test = model.predict(X_test)
        print(f"Validation Performance for cellule {i+1}, famille {fam}:\n",classification_report(y_test, predictions_test))
        
        print(f"accuracy on multilabel labels cellule {i+1} famille {fam} : ",end="")
        predictions_test = model.predict(X_test_m)
        print(sum([y_test_m[f"Selected Resource_{p}.0"].iloc[j] for j, p in enumerate(predictions_test)])/predictions_test.shape[0])


Processing dataset cellule 1, famille 1...
Validation Performance for cellule 1, famille 1:
               precision    recall  f1-score   support

           0       0.74      0.89      0.81       103
           1       0.80      0.58      0.67        76

    accuracy                           0.76       179
   macro avg       0.77      0.74      0.74       179
weighted avg       0.77      0.76      0.75       179

accuracy on multilabel labels cellule 1 famille 1 : 0.7251461988304093

Processing dataset cellule 1, famille 2...
Validation Performance for cellule 1, famille 2:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87       112
           1       0.82      0.76      0.79        70

    accuracy                           0.84       182
   macro avg       0.84      0.82      0.83       182
weighted avg       0.84      0.84      0.84       182

accuracy on multilabel labels cellule 1 famille 2 : 0.8373493975903614

Processing data